## Setup

In [1]:
from google.colab import userdata
access_token = userdata.get('CASM-NER')

In [2]:
%%capture
!pip install transformers
!pip install sentencepiece
!pip install seqeval
!pip install datasets
# !pip install git+https://github.com/ay94/multilingual-ner.git

In [3]:
# from ner import evaluation

In [4]:
## Mount GDrive
from google.colab import drive
drive.mount('/content/drive/', force_remount=True)

## Imports
import os
import sys
import nltk
import time
import torch
import random
import subprocess
import numpy as np
import pandas as pd
import datetime as dt
from itertools import groupby
from tqdm.notebook import tqdm
from datasets import load_dataset
from transformers import pipeline
from collections import Counter, defaultdict
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForTokenClassification, AutoTokenizer
from seqeval.metrics import f1_score as seq_f1, precision_score as seq_precision, recall_score as seq_recall, classification_report as seq_classification
from sklearn.metrics import f1_score as skl_f1, precision_score as skl_precision, recall_score as skl_recall, classification_report as skl_classification

Mounted at /content/drive/


In [5]:
# Append the library files into the notebook system path for import
sys.path.append('/content/drive/Shareddrives/Machine Translation/Model benchmarking/Libraries/1.0.2')
# import custom library files
import ner, utils

## Load datasets

### wikiann

In [6]:
# from ner.dataset_base import HuggingFaceMultilingualDataset
# class Wikiann(HuggingFaceMultilingualDataset):
#     dataset_name = 'wikiann'
#     language = 'ro'
#     license = 'unknown'

# dataset = Wikiann()
# dataset.check_labels()

In [7]:
wikiann_label_map = {
    "O": 0,
    "B-PER": 1,
    "I-PER": 2,
    "B-ORG": 3,
    "I-ORG": 4,
    "B-LOC": 5,
    "I-LOC": 6
}

wikiann = ner.ReadNERData()
wikiann_words, wikiann_labels = wikiann.read_dataset('wikiann', wikiann_label_map, lang='hi')

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/5000 [00:00<?, ? examples/s]

Generating test Split


  0%|          | 0/1000 [00:00<?, ?it/s]

In [8]:
print(ner.check_labels(wikiann_labels))
# Dataset Label Map Alignment to LOC, ORG, PERS, MISC

{'I-LOC', 'I-ORG', 'B-PER', 'B-LOC', 'B-ORG', 'O', 'I-PER'}


## naamapadam

In [9]:
label_map = {
    "O": 0,
    "B-PER": 1,
    "I-PER": 2,
    "B-ORG": 3,
    "I-ORG": 4,
    "B-LOC": 5,
    "I-LOC": 6
}

naamapadam = ner.ReadNERData()
naamapadam_words, naamapadam_labels = naamapadam.read_dataset('ai4bharat/naamapadam', label_map, lang='hi')

Generating train split:   0%|          | 0/985787 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/867 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13460 [00:00<?, ? examples/s]

Generating test Split


  0%|          | 0/867 [00:00<?, ?it/s]

# Evaluate model

In [10]:
alignment = {
'B-organization': 'B-ORG',
'O': 'O',
'B-other': 'O',
'B-person': 'B-PER',
'I-person': 'I-PER',
'B-location': 'B-LOC',
'I-organization': 'I-ORG',
'I-other': 'O',
'I-location': 'I-LOC'
}

model_name = "tner/xlm-roberta-large-conll2003"
model_name_output = 'tner-xlm-roberta-large'
model_evaluation = ner.ModelEvaluation(
    model_name,
    alignment
)

tokenizer_config.json:   0%|          | 0.00/212 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.01k [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

In [11]:
# model_evaluation.model.config.id2label

### wikiann

In [12]:
data_name = "wikiann"
wikiann_evaluation_output = model_evaluation.evaluate_model(wikiann_words, wikiann_labels)

  0%|          | 0/63 [00:00<?, ?it/s]

In [13]:
wikiann_seqeval = wikiann_evaluation_output.get_classification('Seqeval')
wikiann_seqeval

,Tag,Precision,Recall,F1,support
0,LOC,0.4855,0.6473,0.5549,414
1,ORG,0.4758,0.2967,0.3655,364
2,PER,0.7432,0.7911,0.7664,450
3,micro,0.5819,0.5961,0.5889,1228
4,macro,0.5682,0.5784,0.5623,1228
5,weighted,0.5771,0.5961,0.5763,1228


In [14]:
wikiann_sklearn = wikiann_evaluation_output.get_classification('Sklearn')
wikiann_sklearn

,Tag,Precision,Recall,F1,support
0,B-LOC,0.6239,0.8213,0.7091,414
1,B-ORG,0.6972,0.4176,0.5223,364
2,B-PER,0.8017,0.8444,0.8225,450
3,I-LOC,0.6391,0.2714,0.3810,398
4,I-ORG,0.9409,0.3544,0.5149,1123
5,I-PER,0.9204,0.7926,0.8518,598
6,O,0.6974,0.9605,0.8080,2658
7,accuracy,0.7336,6005,None,None
8,macro,0.7601,0.6375,0.6585,6005
9,weighted,0.7640,0.7336,0.7062,6005


### naamapadam

In [15]:
data_name = "naamapadam"
naamapadam_evaluation_output = model_evaluation.evaluate_model(naamapadam_words, naamapadam_labels)

  0%|          | 0/55 [00:00<?, ?it/s]

In [16]:
naamapadam_seqeval = naamapadam_evaluation_output.get_classification('Seqeval')
naamapadam_seqeval

,Tag,Precision,Recall,F1,support
0,LOC,0.7987,0.7948,0.7967,614
1,ORG,0.7752,0.6438,0.7034,525
2,PER,0.8323,0.8734,0.8524,790
3,micro,0.8081,0.7859,0.7968,1929
4,macro,0.8021,0.7707,0.7842,1929
5,weighted,0.8061,0.7859,0.7941,1929


In [17]:
naamapadam_sklearn = naamapadam_evaluation_output.get_classification('Sklearn')
naamapadam_sklearn

,Tag,Precision,Recall,F1,support
0,B-LOC,0.8626,0.8499,0.8562,613
1,B-ORG,0.8548,0.6891,0.7630,521
2,B-PER,0.9150,0.9289,0.9219,788
3,I-LOC,0.8158,0.4673,0.5942,199
4,I-ORG,0.8743,0.6250,0.7289,512
5,I-PER,0.9626,0.8956,0.9279,747
6,O,0.9679,0.9903,0.9790,16513
7,accuracy,0.9574,19893,None,None
8,macro,0.8933,0.7780,0.8244,19893
9,weighted,0.9555,0.9574,0.9551,19893
